# Task 5 — Data Quality & Reconciliation (SkyLedger Airlines)

Post-build **data-quality gate** for the SkyLedger lakehouse (TPC-A practice dataset — synthetic learning data). Runs after the Gold layer is built and validates:

* **DQ-01** — all 8 source tables ingested into Bronze (present, non-empty)
* **DQ-02** — primary keys populated and unique in Silver + Gold dimensions (Bronze raw key health reported as info)
* **DQ-03** — Gold `fact_bookings` has no null surrogate keys and a unique grain
* **DQ-04** — business rules on measures: `ticket_price > 0`, `discount ∈ [0, 1]`, `seats ≥ 1`
* **DQ-05** — Gold revenue and row counts reconcile with Silver (tolerance ≤ 0.01%)
* **DQ-06** — rejected-record counts per table (Silver quarantine) reported

Any **CRITICAL** check failure raises a pipeline-stopping `RuntimeError` in the final cell.

*Geography note:* the fact table carries **both** hierarchies explicitly — passenger geography (`passenger_nation_*` / `passenger_region_*`) and airline geography (`airline_nation_*` / `airline_region_*`). Both sets of keys are checked for nulls.

In [0]:
# ─── Task 5 configuration & check framework ────────────────────────────────────
# Data-quality gate for the SkyLedger lakehouse. Runs AFTER the Gold layer is built.
# Any CRITICAL check failure raises a pipeline-stopping error in the final cell.

CATALOG = "skyledger_airlines"
BRONZE = f"{CATALOG}.bronze"   # raw ingested source tables
SILVER = f"{CATALOG}.silver"   # cleaned tables + <table>_quarantine rejects
GOLD   = f"{CATALOG}.gold"     # fact / dim / kpi tables

RECON_TOLERANCE = 0.0001       # 0.01% relative tolerance for Gold vs Silver

# All eight TPC-A source tables that must exist in Bronze
SOURCE_TABLES = ["passenger", "booking", "flight_booking", "route",
                 "airline_route", "airline", "nation", "region"]

# Primary keys per table (same column names in Bronze and Silver)
PRIMARY_KEYS = {
    "passenger":      ["pa_passengerid"],
    "booking":        ["bk_bookingid"],
    "flight_booking": ["fb_bookingid", "fb_linenumber"],   # composite grain key
    "route":          ["ro_routeid"],
    "airline_route":  ["ar_airlineid", "ar_routeid"],      # composite key
    "airline":        ["al_airlineid"],
    "nation":         ["n_nationkey"],
    "region":        ["r_regionkey"],
}

# Surrogate / degenerate keys that must be populated on every Gold fact row
# (includes BOTH geographies: passenger_* and airline_*)
FACT_SURROGATE_KEYS = ["booking_id", "line_number", "passenger_key", "route_key",
                       "airline_key", "passenger_nation_key", "passenger_region_key",
                       "airline_nation_key", "airline_region_key"]

# Gold dimension primary keys
GOLD_DIM_KEYS = {"dim_passenger": "passenger_key", "dim_route": "route_key",
                 "dim_airline": "airline_key", "dim_geography": "nation_key"}

import pyspark.sql.functions as F

check_results = []   # every check registers here; summarised in the final cell

def record_check(check_id, description, layer, critical, passed, details):
    """Register a check outcome. CRITICAL failures stop the pipeline at the end."""
    check_results.append({"check_id": check_id, "description": description, "layer": layer,
                          "critical": critical, "passed": bool(passed), "details": details})
    status = "PASS" if passed else "FAIL"
    tag = "CRITICAL" if critical else "INFO"
    print(f"[{status} | {tag}] {check_id} — {description}")
    print(f"    {details}\n")

def key_stats(schema, table, keys):
    """Return (row_count, null_key_rows, duplicate_key_values) for a table's key columns."""
    df = spark.table(f"{schema}.{table}")
    total = df.count()
    if total == 0:
        return 0, 0, 0
    nulls = df.filter(" OR ".join(f"{c} IS NULL" for c in keys)).count()
    dups = df.groupBy(*keys).count().filter(F.col("count") > 1).count()
    return total, nulls, dups

def rel_diff(a, b):
    """Relative difference between two numbers (0.0 when both are zero)."""
    a, b = float(a), float(b)
    if b == 0:
        return 0.0 if a == 0 else float("inf")
    return abs(a - b) / abs(b)

print(f"Data-quality gate initialised — catalog: {CATALOG}, reconciliation tolerance: {RECON_TOLERANCE:.4%}")

In [0]:
# CHECK 1 (critical): all eight source tables were ingested into Bronze.
missing_tables, empty_tables, bronze_counts = [], [], {}

for t in SOURCE_TABLES:
    try:
        n = spark.table(f"{BRONZE}.{t}").count()
        bronze_counts[t] = n
        if n == 0:
            empty_tables.append(t)
    except Exception:
        missing_tables.append(t)

dq01_ok = not missing_tables and not empty_tables
record_check(
    "DQ-01", "All 8 source tables ingested into Bronze (present and non-empty)",
    "Bronze", critical=True, passed=dq01_ok,
    details=(f"ingested={len(bronze_counts)}/8"
             + (f", missing={missing_tables}" if missing_tables else "")
             + (f", empty={empty_tables}" if empty_tables else "")),
)

display(spark.createDataFrame(
    [(t, bronze_counts.get(t), "OK" if t in bronze_counts and bronze_counts[t] > 0 else "MISSING/EMPTY")
     for t in SOURCE_TABLES],
    "source_table string, bronze_rows long, status string",
))

In [0]:
# CHECK 2 (critical): primary keys populated and unique in Silver + Gold dimensions.
# Bronze is the RAW landing zone — duplicate/null keys there are EXPECTED in TPC-A dirty
# data and are cleaned during Silver loading (rejects land in <table>_quarantine), so
# Bronze key health is reported as INFO only and never stops the pipeline.

for t, keys in PRIMARY_KEYS.items():
    total, nulls, dups = key_stats(SILVER, t, keys)
    record_check(
        "DQ-02a", f"Silver PK populated & unique — {t} ({', '.join(keys)})",
        "Silver", critical=True, passed=(nulls == 0 and dups == 0),
        details=f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups}",
    )

for dim, key in GOLD_DIM_KEYS.items():
    total, nulls, dups = key_stats(GOLD, dim, [key])
    record_check(
        "DQ-02b", f"Gold dimension key populated & unique — {dim}.{key}",
        "Gold", critical=True, passed=(nulls == 0 and dups == 0),
        details=f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups}",
    )

for t, keys in PRIMARY_KEYS.items():
    total, nulls, dups = key_stats(BRONZE, t, keys)
    record_check(
        "DQ-02c", f"Bronze raw key health (informational) — {t}",
        "Bronze", critical=False, passed=(nulls == 0 and dups == 0),
        details=(f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups} "
                 "(raw rejects are quarantined during Silver cleaning)"),
    )

In [0]:
# CHECK 3 (critical): Gold fact_bookings contains no null surrogate keys,
# and the grain (booking_id + line_number) is unique.
fact = spark.table(f"{GOLD}.fact_bookings")
fact_rows = fact.count()

null_key_expr = " OR ".join(f"{c} IS NULL" for c in FACT_SURROGATE_KEYS)
null_surrogate_rows = fact.filter(null_key_expr).count() if fact_rows else 0

dup_grain = (fact.groupBy("booking_id", "line_number").count()
             .filter(F.col("count") > 1).count()) if fact_rows else 0

record_check(
    "DQ-03a", "fact_bookings has no null surrogate keys (incl. both passenger & airline geography)",
    "Gold", critical=True, passed=(null_surrogate_rows == 0),
    details=f"fact_rows={fact_rows:,}, rows_with_null_surrogate_keys={null_surrogate_rows:,}",
)
record_check(
    "DQ-03b", "fact_bookings grain unique (booking_id + line_number)",
    "Gold", critical=True, passed=(dup_grain == 0),
    details=f"duplicate_grain_values={dup_grain}",
)

In [0]:
# CHECK 4 (critical): numeric measures satisfy the documented business rules
#   ticket_price > 0  |  0 <= discount_rate <= 1  |  seats >= 1
# plus internal consistency: stored net_revenue == CAST(ticket_price * (1 - discount_rate)).
bad_price    = fact.filter("ticket_price IS NULL OR ticket_price <= 0").count()
bad_discount = fact.filter("discount_rate IS NULL OR discount_rate < 0 OR discount_rate > 1").count()
bad_seats    = fact.filter("seats IS NULL OR seats < 1").count()
bad_netrev   = fact.filter(
    "net_revenue <> CAST(ticket_price * (1 - discount_rate) AS DECIMAL(18,2))"
).count()

dq04_ok = all(v == 0 for v in [bad_price, bad_discount, bad_seats, bad_netrev])
record_check(
    "DQ-04", "Business rules: ticket_price>0, discount in [0,1], seats>=1, net_revenue consistent",
    "Gold", critical=True, passed=dq04_ok,
    details=(f"violations: ticket_price={bad_price}, discount_rate={bad_discount}, "
             f"seats={bad_seats}, net_revenue_mismatch={bad_netrev}"),
)

In [0]:
# CHECK 5 (critical): Gold revenue and row counts reconcile with Silver (tolerance <= 0.01%).
# The Silver side is computed INDEPENDENTLY from silver.flight_booking (the source of
# truth for the fact grain) — never from a Gold table — so the reconciliation is real.
gold_rows = fact.count()
silver_rows = spark.table(f"{SILVER}.flight_booking").count()

gold_rev = spark.sql(f"SELECT SUM(net_revenue) AS rev FROM {GOLD}.fact_bookings").collect()[0][0] or 0
silver_rev = spark.sql(f"""
    SELECT SUM(CAST(fb_ticketprice * (1 - fb_discount) AS DECIMAL(18,2))) AS rev
    FROM {SILVER}.flight_booking
""").collect()[0][0] or 0

row_diff = rel_diff(gold_rows, silver_rows)
rev_diff = rel_diff(gold_rev, silver_rev)

record_check(
    "DQ-05a", "Gold fact row count reconciles with Silver flight_booking (<= 0.01%)",
    "Gold<->Silver", critical=True, passed=(row_diff <= RECON_TOLERANCE),
    details=f"gold_rows={gold_rows:,}, silver_rows={silver_rows:,}, rel_diff={row_diff:.6%}",
)
record_check(
    "DQ-05b", "Gold net revenue reconciles with independent Silver calculation (<= 0.01%)",
    "Gold<->Silver", critical=True, passed=(rev_diff <= RECON_TOLERANCE),
    details=(f"gold_net_revenue={float(gold_rev):,.2f}, "
             f"silver_net_revenue={float(silver_rev):,.2f}, rel_diff={rev_diff:.6%}"),
)

display(spark.createDataFrame(
    [("row_count", float(gold_rows), float(silver_rows), row_diff),
     ("net_revenue", float(gold_rev), float(silver_rev), rev_diff)],
    "metric string, gold double, silver double, rel_diff double",
))

In [0]:
# CHECK 6 (reported): rejected-record counts per table.
# For each table: Bronze rows = Silver accepted + Silver quarantine rejects.
# Any gap would mean silent row loss, which is surfaced in the details.
reject_rows = []
for t in SOURCE_TABLES:
    bronze_n = spark.table(f"{BRONZE}.{t}").count()
    silver_n = spark.table(f"{SILVER}.{t}").count()
    quar_n = spark.table(f"{SILVER}.{t}_quarantine").count()
    delta = bronze_n - silver_n
    reject_rows.append((t, bronze_n, silver_n, quar_n, delta))
    record_check(
        "DQ-06", f"Rejected-record accounting — {t} (bronze = silver + quarantine)",
        "Silver", critical=False, passed=(delta == quar_n),
        details=(f"bronze={bronze_n:,}, silver={silver_n:,}, quarantine={quar_n:,}, "
                 f"bronze-silver={delta:,}"
                 + ("" if delta == quar_n else "  <-- MISMATCH: silent row loss!")),
    )

print("Rejected-record counts per table (Silver quarantine):")
display(spark.createDataFrame(
    reject_rows,
    "table string, bronze_rows long, silver_rows long, quarantine_rows long, bronze_minus_silver long",
))

In [0]:
# ─── Final report & pipeline-stopping gate ─────────────────────────────────────
# Any CRITICAL check that failed raises here, stopping the pipeline.
total = len(check_results)
failed = [r for r in check_results if not r["passed"]]
critical_failures = [r for r in failed if r["critical"]]

summary_df = spark.createDataFrame(
    [(r["check_id"], r["layer"], "CRITICAL" if r["critical"] else "INFO",
      "PASS" if r["passed"] else "FAIL", r["description"], r["details"])
     for r in check_results],
    "check_id string, layer string, severity string, status string, description string, details string",
)

print(f"Data-quality checks executed: {total} | passed: {total - len(failed)} | "
      f"failed: {len(failed)} | critical failures: {len(critical_failures)}")
display(summary_df)

if critical_failures:
    lines = [f"  {r['check_id']} [{r['layer']}] {r['description']} — {r['details']}"
             for r in critical_failures]
    raise RuntimeError(
        "DATA QUALITY GATE FAILED — pipeline stopped. "
        f"{len(critical_failures)} critical check(s) violated:\n" + "\n".join(lines)
    )

print("ALL CRITICAL DATA QUALITY CHECKS PASSED — pipeline may proceed.")